<a href="https://colab.research.google.com/github/Argo-Navis-Lab/scalelab/blob/main/notebooks/colab_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ScaleLab on Colab: run one experiment

This notebook only installs ScaleLab and calls `scalelab run`, the same command you run on your laptop. Everything a run writes (checkpoints, artifacts and a copy of the registry) goes to Google Drive, so it outlives the VM.

1. **Runtime → Change runtime type → T4 GPU.**
2. Fill in the settings below, then **Runtime → Run all**.
3. If the session ends before the run does, open this notebook again and **Run all** with the same settings: the run resumes from its last checkpoint (saved every 15 minutes).

In [1]:
import os

# The code to run: a full commit SHA (`git log -1 --format=%H`) pins it exactly.
COMMIT = "2aeaf16b300c6008671f4712cb84b4739a064a3a"  # @param {type:"string"}
EXPERIMENT = "configs/experiments/hello.yaml"  # @param {type:"string"}
OVERRIDES = ""  # @param {type:"string"}

os.environ["COMMIT"] = COMMIT
os.environ["EXPERIMENT"] = EXPERIMENT
os.environ["OVERRIDES"] = OVERRIDES
os.environ["SCALELAB_HOME"] = "/content/drive/MyDrive/scalelab"  # on Drive: outlives the VM

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%%bash
# Install ScaleLab at COMMIT, with exactly the locked dependencies (a few minutes: PyTorch).
set -euo pipefail
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
pip install --quiet uv
rm -rf /content/scalelab
git clone --quiet https://github.com/Argo-Navis-Lab/scalelab.git /content/scalelab
cd /content/scalelab
git checkout --quiet --detach "$COMMIT"
git log -1 --format="code: %H %s"
uv sync --locked --quiet

Tesla T4, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 92.5 MB/s eta 0:00:00
code: 2aeaf16b300c6008671f4712cb84b4739a064a3a m1s23: measure the speed


In [4]:
%%bash
# Run it. The live registry stays on the VM's own disk (SQLite over the Drive mount is
# unreliable), and scalelab copies it to Drive after every change.
cd /content/scalelab
uv run scalelab run "$EXPERIMENT" $OVERRIDES runtime.local_registry=/content/registry.db

experiment:    hello (config c6b6e17b3e7b116b), seed 0
run:           c6b6e17b3e7b116b-s0-2aeaf16b
code:          2aeaf16b (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
resuming (it was running)
done: the last checkpoint was already saved, so there was nothing left to train


In [5]:
%%bash
# Every run in the registry, with its grades.
cd /content/scalelab
uv run python - <<'EOF'
from scalelab.core.registry import Registry

registry = Registry.sqlite("/content/registry.db")
for run in registry.list_runs():
    print(run.run_id, run.status, run.hardware)
    for e in registry.list_evals(run.run_id):
        print(f"  step {e.step:>6}  {e.task}  {e.value:.1%} ± {e.stderr:.1%}  (n={e.n})")
EOF

c6b6e17b3e7b116b-s0-81795f9e done Tesla T4
  step     16  arithmetic-2  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-3  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-4  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-5  0.0% ± 0.0%  (n=20)
c6b6e17b3e7b116b-s0-2aeaf16b done Tesla T4
  step     16  arithmetic-2  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-3  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-4  0.0% ± 0.0%  (n=20)
  step     16  arithmetic-5  0.0% ± 0.0%  (n=20)
47219a7a4a207d7d-s0-2aeaf16b done Tesla T4
  step    100  arithmetic-2  1.0% ± 1.0%  (n=100)
  step    100  arithmetic-3  0.0% ± 0.0%  (n=100)
  step    100  arithmetic-4  0.0% ± 0.0%  (n=100)
  step    100  arithmetic-5  0.0% ± 0.0%  (n=100)
  step    200  arithmetic-2  4.0% ± 2.0%  (n=100)
  step    200  arithmetic-3  1.0% ± 1.0%  (n=100)
  step    200  arithmetic-4  0.0% ± 0.0%  (n=100)
  step    200  arithmetic-5  0.0% ± 0.0%  (n=100)
  step    300  arithmetic-2  4.0% ± 2.0%  (n=100)
  step    300  arithmetic-3  

In [6]:
%%bash
# M1's exit: warm-start m1, m2, m3 and m4 one after another (about 2½ hours on a T4).
# After a disconnection, run it again: finished runs say "already done", and the
# interrupted one resumes from its last checkpoint.
cd /content/scalelab
for model in m1 m2 m3 m4; do
  uv run scalelab run configs/experiments/warmstart.yaml "model=ladder_s/$model" \
    warm_start.tokens=20000000 runtime.local_registry=/content/registry.db
done

experiment:    warmstart (config c73d1d0f4e234964), seed 0
run:           c73d1d0f4e234964-s0-2aeaf16b
code:          2aeaf16b (clean)
deterministic: off
home:          /content/drive/MyDrive/scalelab
registry:      /content/registry.db, copied to home after every change
started
step 1,000: loss 0.092  arithmetic-2 64.3%  arithmetic-3 61.9%  arithmetic-4 59.9%  arithmetic-5 54.7%  (54,982 tokens/s, about 4 min to go)
step 2,000: loss 0.057  arithmetic-2 77.9%  arithmetic-3 64.9%  arithmetic-4 63.0%  arithmetic-5 56.4%  (58,676 tokens/s, about 3 min to go)
step 3,000: loss 0.038  arithmetic-2 88.9%  arithmetic-3 72.3%  arithmetic-4 67.3%  arithmetic-5 64.3%  (58,320 tokens/s, about 71 s to go)
step 3,790: loss 0.025  arithmetic-2 94.5%  arithmetic-3 80.1%  arithmetic-4 69.9%  arithmetic-5 65.3%  (53,824 tokens/s, about 0 s to go)
done: ran 3790 steps in 6 min (20,003,027 tokens and 3.12e+14 FLOPs in all), last loss 0.024546917527914047
experiment:    warmstart (config 71cd3add8c11e667),

In [7]:
%%bash
# The final grades of every warm start, oldest first.
cd /content/scalelab
uv run python - <<'EOF'
import json

from scalelab.core.registry import Registry

registry = Registry.sqlite("/content/registry.db")
for run in registry.list_runs():
    config = json.loads(registry.experiment_json(run.config_hash) or "{}")
    evals = registry.list_evals(run.run_id)
    if config.get("name") != "warmstart" or run.status != "done" or not evals:
        continue
    last = max(e.step for e in evals)
    grades = "  ".join(f"{e.task} {e.value:.1%}" for e in evals if e.step == last)
    tokens = config["warm_start"]["tokens"]
    print(f"{config['model']['name']}  {tokens:>11,} tokens  step {last:>5}:  {grades}")
EOF

m1    2,000,000 tokens  step   379:  arithmetic-2 8.0%  arithmetic-3 4.0%  arithmetic-4 2.0%  arithmetic-5 0.0%
m2    2,000,000 tokens  step   379:  arithmetic-2 56.0%  arithmetic-3 51.0%  arithmetic-4 41.0%  arithmetic-5 30.0%
m3    2,000,000 tokens  step   379:  arithmetic-2 44.0%  arithmetic-3 18.0%  arithmetic-4 2.0%  arithmetic-5 1.0%
m4    2,000,000 tokens  step   379:  arithmetic-2 12.0%  arithmetic-3 1.0%  arithmetic-4 0.0%  arithmetic-5 0.0%
m1   20,000,000 tokens  step  3790:  arithmetic-2 94.5%  arithmetic-3 80.1%  arithmetic-4 69.9%  arithmetic-5 65.3%
m2   20,000,000 tokens  step  3790:  arithmetic-2 98.3%  arithmetic-3 85.6%  arithmetic-4 76.2%  arithmetic-5 66.9%
m3   20,000,000 tokens  step  3790:  arithmetic-2 98.7%  arithmetic-3 90.4%  arithmetic-4 76.8%  arithmetic-5 66.4%
m4   20,000,000 tokens  step  3790:  arithmetic-2 98.3%  arithmetic-3 88.5%  arithmetic-4 77.4%  arithmetic-5 67.8%
